# Analisis Multi-Agent TSLA & GME dengan TradingAgents

Notebook pendamping untuk `Final Assignment - done.ipynb`. Notebook lama menampilkan **harga saham dan revenue historis**; notebook ini menambahkan **keputusan dari tim agent LLM** memakai [TradingAgents](https://github.com/TauricResearch/TradingAgents) (analyst, debat bull/bear, trader, debat risiko, portfolio manager).

> Hanya untuk riset dan belajar, **bukan saran investasi**.

**Prasyarat:** Python 3.11+, `ANTHROPIC_API_KEY` di environment, dan akses internet. Satu analisis memanggil LLM puluhan kali, jadi ada biaya API.

## 1. Instalasi

In [ ]:
# Jalankan sekali. TradingAgents membutuhkan Python >= 3.11.
%pip install -q "git+https://github.com/TauricResearch/TradingAgents.git" yfinance plotly pandas

## 2. Konfigurasi (Claude sebagai LLM)

Key **tidak** ditulis di notebook; isi lewat environment variable.

In [ ]:
import os, getpass
from tradingagents.default_config import DEFAULT_CONFIG

if not os.getenv("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")

config = DEFAULT_CONFIG.copy()
config["llm_provider"] = "anthropic"
config["deep_think_llm"] = "claude-opus-5-5"     # manajer riset & portfolio manager
config["quick_think_llm"] = "claude-sonnet-5-5"  # analyst, researcher, trader
config["max_debate_rounds"] = 1                  # naikkan untuk debat lebih panjang (lebih mahal)
config["max_risk_discuss_rounds"] = 1

## 3. Jalankan analisis

`TRADE_DATE` sebaiknya hari bursa yang sudah lewat.

In [ ]:
from tradingagents.graph.trading_graph import TradingAgentsGraph

TICKERS = ["TSLA", "GME"]
TRADE_DATE = "2026-10-23"

results = {}
for t in TICKERS:
    ta = TradingAgentsGraph(debug=False, config=config)
    state, decision = ta.propagate(t, TRADE_DATE)
    ta.save_reports(state, t)   # laporan lengkap (markdown + HTML) di results_dir
    results[t] = (state, decision)
    print(f"=== {t} ({TRADE_DATE}) ===\n{decision}\n")

## 4. Baca laporan tiap agent

In [ ]:
from IPython.display import Markdown, display

state, _ = results["TSLA"]
for key in ["market_report", "sentiment_report", "news_report", "fundamentals_report", "trader_investment_plan", "final_trade_decision"]:
    if key in state:
        display(Markdown(f"### {key}\n\n{state[key]}"))

## 5. Gabungkan dengan grafik harga

Harga historis dari yfinance, seperti notebook tugas, dengan tanggal keputusan agent ditandai.

In [ ]:
import yfinance as yf, pandas as pd
import plotly.graph_objects as go

fig = go.Figure()
for t in TICKERS:
    h = yf.Ticker(t).history(period="2y").reset_index()
    fig.add_trace(go.Scatter(x=pd.to_datetime(h["Date"]).dt.tz_localize(None), y=h["Close"], name=t))
fig.add_vline(x=pd.Timestamp(TRADE_DATE), line_dash="dash", annotation_text="Tanggal keputusan agent")
fig.update_layout(title="Harga penutupan 2 tahun terakhir", yaxis_title="USD", height=450)
fig.show()

## 6. (Opsional) Backtest

TradingAgents menyediakan `run_backtest` untuk menilai keputusan agent terhadap pergerakan harga setelahnya. Lihat `tradingagents/backtest.py` dan README repo untuk parameternya; backtest memanggil LLM di setiap tanggal sehingga biayanya naik cepat.